# Hands-on 21: Menyiapkan Dataset Regresi Prediksi Konsumsi Energi

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 7.

Target `target_power_next_1step` = daya 10 menit ke depan. Fitur: waktu, encoding siklik,
lag (1/2/6), rolling (MA-3/6, std-6), difference. Split time-aware 70/15/15.
Output: `data/processed/regression_energy/`.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Siapkan Dataset

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/prepare_regression_energy_dataset.py"], capture_output=True, text=True)
print(r.stdout)

=== Ringkasan Dataset Regresi ===
Total data : 4313
Train      : 3019
Validation : 647
Test       : 647

Preview:
            timestamp  power_watt  target_power_next_1step
0 2026-01-01 01:00:00      102.36                   123.43
1 2026-01-01 01:10:00      123.43                   113.13
2 2026-01-01 01:20:00      113.13                   106.13
3 2026-01-01 01:30:00      106.13                   104.24
4 2026-01-01 01:40:00      104.24                    84.76
5 2026-01-01 01:50:00       84.76                   124.31
6 2026-01-01 02:00:00      124.31                   108.88
7 2026-01-01 02:10:00      108.88                   128.93
8 2026-01-01 02:20:00      128.93                   111.64
9 2026-01-01 02:30:00      111.64                   121.46

File disimpan di: data/processed/regression_energy



## 2. Verifikasi Output

In [3]:
import pandas as pd
out = Path("data/processed/regression_energy")
for f in sorted(out.glob("*.csv")):
    d = pd.read_csv(f)
    print(f"{f.name:15s} -> {d.shape}")
X_train = pd.read_csv(out / "X_train.csv")
display(X_train.head())

X_test.csv      -> (647, 14)
X_train.csv     -> (3019, 14)
X_val.csv       -> (647, 14)
energy_linear_predictions.csv -> (647, 2)
energy_rf_predictions.csv -> (647, 2)
full_dataset.csv -> (4313, 17)
pred_linear_test.csv -> (647, 2)
pred_rf_test.csv -> (647, 2)
pred_ridge_test.csv -> (647, 2)
test.csv        -> (647, 17)
test_predictions.csv -> (647, 2)
train.csv       -> (3019, 17)
val.csv         -> (647, 17)
validation.csv  -> (647, 17)
y_test.csv      -> (647, 1)
y_train.csv     -> (3019, 1)
y_val.csv       -> (647, 1)


,power_watt,hour,day_of_week,is_weekend,hour_sin,hour_cos,power_lag_1,power_lag_2,power_lag_6,power_ma_3,power_ma_6,power_std_6,power_diff_1,power_diff_6
0,102.36,1,3,0,0.258819,0.965926,110.00,114.11,98.13,108.823333,113.948333,9.943029,-7.64,4.23
1,123.43,1,3,0,0.258819,0.965926,102.36,110.00,112.51,111.930000,115.768333,10.604507,21.07,10.92
2,113.13,1,3,0,0.258819,0.965926,123.43,102.36,132.38,112.973333,112.560000,6.804863,-10.30,-19.25
3,106.13,1,3,0,0.258819,0.965926,113.13,123.43,112.33,114.230000,111.526667,7.299536,-7.00,-6.20
4,104.24,1,3,0,0.258819,0.965926,106.13,113.13,114.11,107.833333,109.881667,7.701972,-1.89,-9.87


## Penutup

Dataset regresi siap: 4313 baris, 14 fitur, target daya 10 menit ke depan. Dipakai pada
Hands-on 23 (regresi Linear/Ridge/Random Forest) dan Hands-on 25 Bab 10 (forecasting
energi harian).

## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 18 dalam urutan buku) sebagai panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 6. Hal penting yang harus dipahami

Mengapa target digeser ke atas dengan shift(-1)?
Karena kita ingin nilai masa depan menjadi label target untuk fitur saat ini.

Mengapa tidak boleh sembarang shuffle?
Karena ini time series. Model harus diuji pada data yang benar-benar lebih baru.

Mengapa ada lag dan rolling?
Karena prediksi energi jangka pendek sangat dipengaruhi histori beberapa langkah
sebelumnya. Ini juga sejalan dengan literatur time series dan prediksi energi IoT.

## Jawaban dari data: Hal penting yang harus dipahami (Hands-on 18): jawaban dari data

In [4]:
import pandas as pd
tr = pd.read_csv("data/processed/regression_energy/train.csv")
te = pd.read_csv("data/processed/regression_energy/test.csv")
print("1. Target digeser shift(-1): target_next baris i sama dengan power_watt baris i+1 ->",
      bool((tr["power_watt"].shift(-1).iloc[:-1].values == tr["target_power_next_1step"].iloc[:-1].values).all()))
tr_t, te_t = pd.to_datetime(tr["timestamp"]), pd.to_datetime(te["timestamp"])
print(f"2. Split berbasis waktu: train berakhir {tr_t.max()}, test mulai {te_t.min()} -> test lebih baru: {te_t.min() > tr_t.max()}")
print("3. Fitur lag dan rolling yang tersedia:", [c for c in tr.columns if "lag" in c or "ma_" in c or "std_" in c])

1. Target digeser shift(-1): target_next baris i sama dengan power_watt baris i+1 -> True
2. Split berbasis waktu: train berakhir 2026-01-22 00:00:00, test mulai 2026-01-26 12:00:00 -> test lebih baru: True
3. Fitur lag dan rolling yang tersedia: ['power_lag_1', 'power_lag_2', 'power_lag_6', 'power_ma_3', 'power_ma_6', 'power_std_6']
